# **Explanation metrics: one image, four maps, three checks**

Practice for the module [“Evaluating the Quality of Explanations”](https://ai-interpretability.school/learn/17).

In the module we covered several ways of asking a saliency map whether it tells the truth about the model. Here
we put them side by side on one image and one model:

- we build four maps: Grad-CAM, Integrated Gradients, SmoothGrad and a random one;
- we compute Faithfulness Correlation from the Quantus library;
- we build ROAD curves — removal of important pixels with filling from the neighbors, without retraining;
- we run a sanity check — randomization of the weights of the top blocks of the network;
- we collect the results in a “method × metric” table with ranks and see whether the order of the methods agrees.

Piggy photo: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), Pexels License.

In [ ]:
!pip install -q quantus==0.6.0

In [ ]:
import copy
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import quantus
from io import BytesIO
from PIL import Image
from scipy.stats import spearmanr
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from quantus.functions.perturb_func import noisy_linear_imputation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Model and image

ResNet-50 with ImageNet weights and a photo from the course's open repository — the same pair as in the practices
of the modules “Heuristic Gradient-based Explanations” and “Localization-based Explanations and CAM”. `eval()` mode is required: in training mode BatchNorm computes the
statistics over a batch of one image, and the prediction no longer matches the real one.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
image = Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/piglet.jpg').content)).convert('RGB')
x = transform(image).unsqueeze(0).to(device)
categories = requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()

with torch.no_grad():
    probs = F.softmax(model(x), dim=1)[0]
target = probs.argmax().item()
print(f'class {target} “{categories[target]}”, probability {probs[target].item():.3f}')

## 1. Four maps

Grad-CAM and SmoothGrad are given ready-made: the notebook needs their maps, not the inner workings
of the methods. The random map does not depend on the model at all: it is the point of comparison that a method should
not be worse than. All random noise is generated on the CPU with a fixed seed, so the maps are the same on the CPU
and on the GPU.

SmoothGrad here averages the absolute values of the gradients over the noisy copies, like `captum.attr.NoiseTunnel` over `Saliency` with `abs=True`. Smilkov et al. (2017) average the gradients themselves and take the absolute value only for the picture; these are two different maps.

In [ ]:
def gradient(model, x, target):
    """Gradient of the class logit with respect to the input."""
    x = x.clone().requires_grad_(True)
    grad, = torch.autograd.grad(model(x)[0, target], x)
    return grad

def grad_cam(model, x, target):
    """Grad-CAM on the output of layer4, upsampled to the image size."""
    store = {}
    hook = model.layer4.register_forward_hook(lambda module, inp, out: store.update(act=out))
    logit = model(x)[0, target]
    hook.remove()
    grad, = torch.autograd.grad(logit, store['act'])
    cam = F.relu((grad.mean(dim=(2, 3), keepdim=True) * store['act']).sum(1, keepdim=True))
    return F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0].detach()

def smoothgrad(model, x, target, sigma_frac=0.15, n=25):
    """SmoothGrad: the mean |gradient| map over n noisy copies of the input."""
    gen = torch.Generator().manual_seed(0)
    sigma = sigma_frac * (x.max() - x.min())
    acc = torch.zeros(x.shape[-2:], device=x.device)
    for _ in range(n):
        noise = torch.randn(x.shape, generator=gen).to(x.device)
        acc += gradient(model, x + noise * sigma, target).abs().max(dim=1).values[0]
    return (acc / n).detach()

def random_map(model, x, target):
    """Random map: does not depend on the model."""
    return torch.rand(x.shape[-2:], generator=torch.Generator().manual_seed(0)).to(x.device)

**Exercise 1.** Write Integrated Gradients with a zero reference input (after normalization zero is the mean
ImageNet color) and $m$ steps along the path: the mean of the gradients at the points $\alpha x$,
$\alpha = 1/m, 2/m, \dots, 1$, multiplied by $x$ and summed over the channels. Check the completeness axiom: the
sum of the map should be close to the difference of the logits $f(x) - f(0)$. Compute the sum of the map for
$m = 32$ and for $m = 128$ and compare it with the difference of the logits. The rest of the notebook uses
$m = 128$. Here the integral is computed as a right Riemann sum, as in Sundararajan et al. (2017); `captum.attr.IntegratedGradients` uses Gauss–Legendre nodes by default (`method='gausslegendre'`, `n_steps=50`), and the sum of its map at the same $m$ comes out different.

In [ ]:
# Your code here

**Your answer:**

In [ ]:
methods = {'Grad-CAM': grad_cam, 'IG': integrated_gradients, 'SmoothGrad': smoothgrad, 'Random': random_map}
maps = {name: method(model, x, target) for name, method in methods.items()}

shown = transforms.Resize((224, 224))(image)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(shown)
axes[0].set_title(f'Input: {categories[target]}')
for ax, (name, m) in zip(axes[1:], maps.items()):
    ax.imshow(m.abs().cpu().numpy(), cmap='hot')
    ax.set_title(name)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Faithfulness Correlation in Quantus

The metric from [Bhatt et al., 2020](https://arxiv.org/abs/2005.00631): a random subset of pixels $S$ is chosen,
its values are replaced with a baseline, and two quantities are computed — the total importance of the subset's
pixels $\sum_{i \in S} a_i$ and the drop of the model output $f(x) - f(x_{S \leftarrow \text{baseline}})$. After
repeating this many times, the Pearson correlation between the two quantities is taken. The higher it is, the
better the map predicts which subset will affect the answer more.

In Quantus this is `quantus.FaithfulnessCorrelation`; by default $|S| = 224$ pixels out of $224 \times 224$ and 100
repetitions. The subsets are drawn with `np.random`, so the result depends on the seed.

**Exercise 2.** Compute Faithfulness Correlation for the four maps with the default settings (black baseline,
100 repetitions, $|S| = 224$) with the seeds `np.random.seed(0)` … `np.random.seed(4)`. Print the mean and the
standard deviation over the five seeds for every map. Does the difference between the maps stand out from the
spread between the seeds?

In [ ]:
# Your code here

**Your answer:**

## 3. ROAD: removing important pixels with filling from the neighbors

ROAD ([Rong et al., 2022](https://arxiv.org/abs/2202.00449)) removes a fraction of the pixels in the order of the
map and fills the removed pixels with the `noisy_linear_imputation` function from Quantus: every removed pixel gets a weighted
mean of its neighbors plus a little noise. The model is not retrained. Pixels can be removed in two orders:

- MoRF (most relevant first) — the most important pixels first; a good map lowers the class probability quickly,
  the lower the mean probability over the curve, the better;
- LeRF (least relevant first) — the least important first; a good map keeps the class probability for long, the
  higher the mean probability over the curve, the better.

The curve is the class probability after removing 5%, 10%, …, 95% of the pixels; the summary of the curve is the
mean over the 19 points.

In [ ]:
fractions = np.arange(5, 100, 5) / 100

def road_curve(m, order):
    """Class probability after removing a fraction of the pixels in MoRF or LeRF order."""
    importance = m.cpu().numpy().flatten()
    ranked = np.argsort(-importance) if order == 'MoRF' else np.argsort(importance)
    np.random.seed(0)                                   # the filling noise is reproducible
    curve = []
    for frac in fractions:
        filled = noisy_linear_imputation(x_np[0].copy(), ranked[:int(frac * importance.size)], noise=0.01)
        with torch.no_grad():
            out = F.softmax(model(torch.tensor(filled[None], dtype=torch.float32, device=device)), dim=1)
        curve.append(out[0, target].item())
    return np.array(curve)

**Exercise 3.** Build the MoRF and LeRF curves for the four maps and plot them. Compute the mean class probability
over every curve. Which map is the best by MoRF and which by LeRF? Does the order of the four maps agree between
MoRF and LeRF?

In [ ]:
# Your code here

**Your answer:**

## 4. Sanity check: weight randomization

The check from [Adebayo et al., 2018](https://arxiv.org/abs/1810.03292): the weights of the network are
re-initialized cumulatively from the output toward the input — first `fc`, then `fc` and `layer4`, and so on — and
the map is built again. The similarity measure is the Spearman correlation between the original map and the new
one, both flattened into a vector. A map that explains the model must change: the closer the correlation is to
zero, the better, so we will compare the maps by the absolute value of the correlation. The weights are re-initialized on the CPU with a fixed seed, so the broken network is the same on the CPU
and on the GPU.

In [ ]:
def randomize(model, blocks, seed=0):
    """A copy of the network with randomly re-initialized weights in the listed blocks."""
    broken = copy.deepcopy(model).cpu()
    torch.manual_seed(seed)
    for block in blocks:
        for module in getattr(broken, block).modules():
            if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
                module.reset_parameters()
    return broken.to(device).eval()

cascade = [['fc'], ['fc', 'layer4'], ['fc', 'layer4', 'layer3'], ['fc', 'layer4', 'layer3', 'layer2']]

**Exercise 4.** For every stage of the cascade and every one of the four maps compute the Spearman correlation
between the map of the trained network and the map of the broken network. What is the correlation for SmoothGrad
after re-initializing only `fc` (round to two decimal places)? Why is the correlation of the random map equal to 1 at
every stage, and what happens to Grad-CAM at the last stage?

In [ ]:
# Your code here

**Your answer:**

## 5. The “method × metric” table

**Exercise 5.** Collect three metrics in one table: the mean Faithfulness Correlation over the five seeds (higher is
better), the mean class probability over the ROAD MoRF curve (lower is better) and the absolute Spearman correlation
after re-initializing `fc` and `layer4` (lower is better). Give every map a rank by every metric (1 is the best). Does the
order of the maps agree across the three metrics?

In [ ]:
# Your code here

**Your answer:**

## What to take away from the notebook

- **The order of the methods depends on the metric.** On one image and one model Grad-CAM is the best of the four
  maps by ROAD MoRF, the last by ROAD LeRF and third by the sanity check.
- **The random map is a required point of comparison.** Only it shows whether a metric tells a method from guessing:
  Faithfulness Correlation with the default settings put the random map above two methods.
- **One image is noise.** The spread of Faithfulness Correlation between the seeds is of the same order as the
  difference between the maps, and the ROAD curves are non-monotonic. Conclusions about a method are drawn from a set
  of images and several seeds.
- **A sanity check is run as a cascade.** SmoothGrad survived the re-initialization of `fc` with a correlation of
  0.77 and changed only after `layer4` — this is visible only at separate stages.